# Functions and Tools

From the [Functions documentation](../../../docs/source/workflows/functions/index.md):

> *"Functions (tools) are the main building blocks of NeMo Agent toolkit and define the logic of your workflow."*

Functions are type-safe, asynchronous operations with:
- Schema-based input/output validation via Pydantic models
- Support for both single and streaming outputs
- Unified interfaces to improve composability

## Function Types

| Type | Documentation | Description |
|------|---------------|-------------|
| **Functions** | [Functions](../../../docs/source/workflows/functions/index.md) | Individual tools with defined input/output schemas |
| **Function Groups** | [Function Groups](../../../docs/source/workflows/function-groups.md) | Package related functions to share configuration, context, and resources |
| **MCP Client** | [MCP](../../../docs/source/workflows/mcp/index.md) | Connect to tools served by remote MCP servers |
| **A2A Client** | [A2A](../../../docs/source/workflows/a2a/index.md) | Connect to and interact with remote A2A agents |

### Function Groups

From the [Function Groups documentation](../../../docs/source/workflows/function-groups.md):

> *"Function groups let you package multiple related functions together so they can share configuration, context, and resources."*

Function groups solve common issues like duplicated configuration, resource waste, and inconsistent state.


In [ ]:
import getpass
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - examples will fail")


✅ Environment configured


## 1. Built-in Functions

NAT includes several built-in functions you can use immediately:


In [2]:
from nat.tool.datetime_tools import CurrentTimeTool

# CurrentTimeTool - Returns current date and time
time_tool = CurrentTimeTool(
    name="current_time",
    # Optional: customize timezone
    # timezone="America/New_York"
)

print(f"✅ Created: {time_tool.computed_name}")


✅ Created: current_time


## 2. Function Groups

Function groups package related functions together so they can share configuration, context, and resources. The calculator example demonstrates this:


In [3]:
# Function groups from installed example packages
try:
    from nat_simple_calculator.register import CalculatorToolGroup

    calculator = CalculatorToolGroup(
        name="calculator",
        # Can specify which functions to include
        include=["add", "subtract", "multiply", "divide"],
    )
    print("✅ Calculator group created with functions: add, subtract, multiply, divide")
except ImportError:
    print("⚠️  Install with: uv pip install -e examples/getting_started/simple_calculator")


✅ Calculator group created with functions: add, subtract, multiply, divide


## 3. MCP Client (Model Context Protocol)

MCP allows you to connect to external servers that expose functions. This is useful for:
- Using functions from other services
- Connecting to local MCP servers
- Accessing authenticated APIs


In [4]:
from pydantic import HttpUrl

from nat.plugins.mcp.client_config import MCPClient
from nat.plugins.mcp.client_config import MCPServerConfig
from nat.plugins.mcp.client_config import MCPToolOverrideConfig

# Example 1: Local MCP server (stdio transport)
mcp_time = MCPClient(
    server=MCPServerConfig(
        transport="stdio",
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/Los_Angeles"],
    ),
    tool_overrides={
        "get_current_time": MCPToolOverrideConfig(
            alias="get_time_mcp",  # Rename the tool
            description="Get current time from MCP server",
        ),
    },
    name="mcp_time",
)

print("✅ MCP Client (stdio) configured")


✅ MCP Client (stdio) configured


In [5]:
# Example 2: Remote MCP server (HTTP transport)
mcp_remote = MCPClient(
    server=MCPServerConfig(
        transport="streamable-http",
        url=HttpUrl("http://localhost:9901/mcp"),  # NAT MCP server
    ),
    include=["calculator.add", "calculator.multiply"],  # Only include specific tools
    name="mcp_remote",
)

print("✅ MCP Client (HTTP) configured")


✅ MCP Client (HTTP) configured


## 4. A2A Client (Agent-to-Agent)

A2A allows your agent to use other agents as functions. Since agents are functions themselves, this enables sophisticated hierarchical agent orchestration:


In [6]:
from datetime import timedelta

try:
    from nat.plugins.a2a.client.a2a_client import A2AClient

    # Connect to an A2A agent server
    calculator_agent = A2AClient(
        url=HttpUrl("http://localhost:10000"),
        task_timeout=timedelta(seconds=60),
        include_skills_in_description=True,  # Include agent's capabilities
        name="calculator_agent",
    )
    print("✅ A2A Client configured")
except ImportError:
    print("⚠️  Install with: uv pip install -e packages/nvidia_nat_a2a")


✅ A2A Client configured


## 5. Combining Functions in an Agent


In [7]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    name="nim_llm",
)

# Combine different tool types
tools = [
    time_tool,      # Built-in function
    mcp_time,       # MCP client
]

agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print(f"✅ Agent created with {len(tools)} tools")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Agent created with 2 tools


In [8]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "tools_example.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Saved to: {config_path}")
print("\n" + "=" * 50 + "\n")
with open(config_path) as f:
    print(f.read())


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


📄 Saved to: configs/tools_example.yaml


functions:
  current_time:
    _type: current_datetime

function_groups:
  mcp_time:
    _type: mcp_client
    server:
      transport: stdio
      url: null
      command: python
      args:
      - -m
      - mcp_server_time
      - --local-timezone=America/Los_Angeles
      env: null
      auth_provider: null
    tool_overrides:
      get_current_time:
        alias: get_time_mcp
        description: Get current time from MCP server

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time
  - mcp_time



## CLI Commands

```bash
# Run the workflow
nat run --config_file configs/tools_example.yaml --input "What time is it?"

# Start an MCP server to connect to
nat mcp serve --config_file examples/getting_started/simple_calculator/configs/config.yml

# Start an A2A server to connect to
nat a2a serve --config_file examples/getting_started/simple_calculator/configs/config.yml --port 10000
```

## Summary

✅ **Built-in functions** - Ready-to-use functions like CurrentTimeTool  
✅ **Function groups** - Collections of related functions sharing configuration  
✅ **MCP Client** - Functions exposed by external MCP servers  
✅ **A2A Client** - Use other agents as functions  

## Next Steps

- **[05_llms.ipynb](./05_llms.ipynb)** - Configure different LLM providers
- **[06_memory.ipynb](./06_memory.ipynb)** - Add memory to your agents
